In [1]:
import pandas as pd
import numpy as np
import os
import time
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Dynamically locate Cyber dataset directory
candidate_paths = ["Cyber", "cyber-pack (4)/Cyber", "dataset/Cyber"]
data_base_dir = next((p for p in candidate_paths if os.path.exists(os.path.join(p, "training", "flows.parquet"))), "Cyber")

train_path = os.path.join(data_base_dir, "training", "flows.parquet")
eval_path = os.path.join(data_base_dir, "eval", "flows.parquet")
print(f"Dataset located at: {data_base_dir}")

train_df = pd.read_parquet(train_path)
eval_df = pd.read_parquet(eval_path)

print("Training shape:", train_df.shape)
print("Evaluation shape:", eval_df.shape)

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nEvaluation labels:")
print(eval_df["label"].value_counts())


Dataset located at: cyber-pack (4)/Cyber
Training shape: (456737, 80)
Evaluation shape: (114185, 80)

Training labels:
label
BENIGN         240000
DOS             96000
PORT_SCAN       57600
DDOS            57600
BRUTE_FORCE      5537
Name: count, dtype: int64

Evaluation labels:
label
BENIGN         60001
DOS            24000
PORT_SCAN      14400
DDOS           14400
BRUTE_FORCE     1384
Name: count, dtype: int64


In [3]:
TARGET = "label"
DROP_COLS = ["label", "record_id"]

assert TARGET in train_df.columns
assert TARGET in eval_df.columns

feature_cols = [
    col for col in train_df.columns
    if col not in DROP_COLS
]

assert set(feature_cols).issubset(eval_df.columns)

X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET].copy()

X_eval = eval_df[feature_cols].copy()
y_eval = eval_df[TARGET].copy()

# Convert infinite values to missing values
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_eval = X_eval.replace([np.inf, -np.inf], np.nan)

# Ensure numeric features
assert all(pd.api.types.is_numeric_dtype(X_train[c]) for c in feature_cols)

print("Features:", len(feature_cols))
print("Training samples:", len(X_train))
print("Evaluation samples:", len(X_eval))
print("Labels:", sorted(y_train.unique()))

Features: 78
Training samples: 456737
Evaluation samples: 114185
Labels: ['BENIGN', 'BRUTE_FORCE', 'DDOS', 'DOS', 'PORT_SCAN']


In [4]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=30,
        max_depth=20,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

start = time.time()

model.fit(X_train, y_train)

training_time = time.time() - start

print(f"Training completed in {training_time:.2f} seconds!")

Training completed in 33.54 seconds!


In [5]:
y_pred = model.predict(X_eval)

accuracy = accuracy_score(y_eval, y_pred)
macro_f1 = f1_score(
    y_eval, y_pred,
    average="macro",
    zero_division=0
)

print("Accuracy:", round(accuracy, 4))
print("Macro F1:", round(macro_f1, 4))

print("\nClassification Report:")
print(classification_report(y_eval, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_eval, y_pred))

Accuracy: 1.0
Macro F1: 0.9999

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     60001
 BRUTE_FORCE       1.00      1.00      1.00      1384
        DDOS       1.00      1.00      1.00     14400
         DOS       1.00      1.00      1.00     24000
   PORT_SCAN       1.00      1.00      1.00     14400

    accuracy                           1.00    114185
   macro avg       1.00      1.00      1.00    114185
weighted avg       1.00      1.00      1.00    114185


Confusion Matrix:
[[60001     0     0     0     0]
 [    1  1383     0     0     0]
 [    0     0 14400     0     0]
 [    2     0     0 23998     0]
 [    2     0     0     0 14398]]


In [6]:
# ===================================================================
# THREAT & CONFIDENCE PREDICTION ENGINE
# ===================================================================
# In this 5-class CICIDS2017 dataset:
# - BENIGN: Normal network traffic (Non-threat)
# - DOS, DDOS, PORT_SCAN, BRUTE_FORCE: Malicious activities (Threats)
NON_THREAT_CLASS = "BENIGN"
THREAT_CLASSES = [c for c in model.classes_ if c != NON_THREAT_CLASS]

print("Classes in model:", list(model.classes_))
print("Non-Threat Class:", NON_THREAT_CLASS)
print("Threat Classes:", THREAT_CLASSES)

def predict_threats(model, X_features, threshold=0.5):
    """
    Predicts whether flows are threats, the specific threat class, and confidence level.
    
    Returns pd.DataFrame with:
    - predicted_class: Most likely class
    - confidence: Model probability for the predicted class (0.0 to 1.0)
    - is_threat: True if classified as an attack with threat_score >= threshold
    - threat_score: Overall attack probability (1.0 - P(BENIGN))
    - threat_type: Attack type if is_threat is True, else 'None'
    """
    proba = model.predict_proba(X_features)
    classes = model.classes_
    benign_idx = list(classes).index(NON_THREAT_CLASS)
    
    pred_indices = np.argmax(proba, axis=1)
    predicted_classes = classes[pred_indices]
    confidences = np.max(proba, axis=1)
    
    # Threat score is total probability of being any threat class
    threat_scores = 1.0 - proba[:, benign_idx]
    
    # A flow is flagged as a threat if it is predicted as an attack class
    is_threat = (predicted_classes != NON_THREAT_CLASS) & (threat_scores >= threshold)
    threat_types = np.where(is_threat, predicted_classes, "None")
    
    results = pd.DataFrame({
        "predicted_class": predicted_classes,
        "confidence": confidences,
        "is_threat": is_threat,
        "threat_score": threat_scores,
        "threat_type": threat_types
    })
    return results

eval_predictions = predict_threats(model, X_eval)
print("\nSample predictions on Evaluation Set:")
print(eval_predictions.head(30))


Classes in model: ['BENIGN', 'BRUTE_FORCE', 'DDOS', 'DOS', 'PORT_SCAN']
Non-Threat Class: BENIGN
Threat Classes: ['BRUTE_FORCE', 'DDOS', 'DOS', 'PORT_SCAN']

Sample predictions on Evaluation Set:
   predicted_class  confidence  is_threat  threat_score threat_type
0           BENIGN         1.0      False           0.0        None
1           BENIGN         1.0      False           0.0        None
2           BENIGN         1.0      False           0.0        None
3           BENIGN         1.0      False           0.0        None
4           BENIGN         1.0      False           0.0        None
5           BENIGN         1.0      False           0.0        None
6           BENIGN         1.0      False           0.0        None
7           BENIGN         1.0      False           0.0        None
8           BENIGN         1.0      False           0.0        None
9           BENIGN         1.0      False           0.0        None
10          BENIGN         1.0      False           0.0 

In [7]:
def predict_single_flow(model, flow_record, feature_columns):
    """
    Predict threat class and confidence level for a single network flow record.
    Accepts a dict, pd.Series, or 1-row DataFrame.
    """
    if isinstance(flow_record, dict):
        flow_df = pd.DataFrame([flow_record])[feature_columns]
    elif isinstance(flow_record, pd.Series):
        flow_df = pd.DataFrame([flow_record.to_dict()])[feature_columns]
    else:
        flow_df = flow_record[feature_columns]
        
    flow_clean = flow_df.replace([np.inf, -np.inf], np.nan)
    proba = model.predict_proba(flow_clean)[0]
    classes = model.classes_
    
    pred_idx = np.argmax(proba)
    pred_class = classes[pred_idx]
    confidence = proba[pred_idx]
    
    benign_idx = list(classes).index(NON_THREAT_CLASS)
    threat_score = 1.0 - proba[benign_idx]
    is_threat = (pred_class != NON_THREAT_CLASS)
    
    return {
        "is_threat": is_threat,
        "threat_type": pred_class if is_threat else "None",
        "confidence": float(confidence),
        "threat_score": float(threat_score),
        "probabilities": {cls_name: round(float(prob), 4) for cls_name, prob in zip(classes, proba)}
    }

print("=== Testing Single Flow Prediction for each of the 5 classes ===")
for target_cls in model.classes_:
    sample_idx = eval_df[eval_df["label"] == target_cls].index[0]
    sample_row = eval_df.loc[sample_idx]
    
    pred_res = predict_single_flow(model, sample_row, feature_cols)
    status_icon = "🚨 THREAT" if pred_res["is_threat"] else "🛡️ BENIGN"
    print(f"\n[Actual: {target_cls}] -> {status_icon}")
    print(f"  Threat Type  : {pred_res['threat_type']}")
    print(f"  Confidence   : {pred_res['confidence'] * 100:.2f}%")
    print(f"  Threat Score : {pred_res['threat_score']:.4f}")
    print(f"  Probabilities: {pred_res['probabilities']}")


=== Testing Single Flow Prediction for each of the 5 classes ===

[Actual: BENIGN] -> 🛡️ BENIGN
  Threat Type  : None
  Confidence   : 100.00%
  Threat Score : 0.0000
  Probabilities: {'BENIGN': 1.0, 'BRUTE_FORCE': 0.0, 'DDOS': 0.0, 'DOS': 0.0, 'PORT_SCAN': 0.0}

[Actual: BRUTE_FORCE] -> 🚨 THREAT
  Threat Type  : BRUTE_FORCE
  Confidence   : 96.67%
  Threat Score : 0.9667
  Probabilities: {'BENIGN': 0.0333, 'BRUTE_FORCE': 0.9667, 'DDOS': 0.0, 'DOS': 0.0, 'PORT_SCAN': 0.0}

[Actual: DDOS] -> 🚨 THREAT
  Threat Type  : DDOS
  Confidence   : 100.00%
  Threat Score : 1.0000
  Probabilities: {'BENIGN': 0.0, 'BRUTE_FORCE': 0.0, 'DDOS': 1.0, 'DOS': 0.0, 'PORT_SCAN': 0.0}

[Actual: DOS] -> 🚨 THREAT
  Threat Type  : DOS
  Confidence   : 99.45%
  Threat Score : 0.9945
  Probabilities: {'BENIGN': 0.0055, 'BRUTE_FORCE': 0.0, 'DDOS': 0.0, 'DOS': 0.9945, 'PORT_SCAN': 0.0}

[Actual: PORT_SCAN] -> 🚨 THREAT
  Threat Type  : PORT_SCAN
  Confidence   : 100.00%
  Threat Score : 1.0000
  Probabilities: {'BE

In [8]:
# Confidence breakdown per class
summary_stats = eval_predictions.groupby("predicted_class").agg(
    total_flows=("confidence", "count"),
    mean_confidence=("confidence", "mean"),
    min_confidence=("confidence", "min"),
    max_confidence=("confidence", "max"),
    threat_detected=("is_threat", "sum")
).reset_index()

print("Threat Classification & Confidence Breakdown on Evaluation Set:")
print(summary_stats.to_string(index=False))


Threat Classification & Confidence Breakdown on Evaluation Set:
predicted_class  total_flows  mean_confidence  min_confidence  max_confidence  threat_detected
         BENIGN        60006         0.999606        0.541061             1.0                0
    BRUTE_FORCE         1383         0.998977        0.564473             1.0             1383
           DDOS        14400         0.999963        0.910619             1.0            14400
            DOS        23998         0.999668        0.591187             1.0            23998
      PORT_SCAN        14398         0.999863        0.787071             1.0            14398


In [9]:
os.makedirs("models", exist_ok=True)
model_path = "models/threat_classifier.joblib"

joblib.dump(
    {
        "model": model,
        "feature_columns": feature_cols,
        "classes": list(model.classes_),
        "threat_classes": THREAT_CLASSES,
        "non_threat_class": NON_THREAT_CLASS,
        "threshold": 0.5
    },
    model_path
)

print(f"Model and complete threat metadata saved successfully to {model_path}!")


Model and complete threat metadata saved successfully to models/threat_classifier.joblib!
